# Cross-Domain Scenario
## Description

We start by loading three files: book tags with scores, movie tags with scores, and a table that tells us which book tags match which movie tags.  We then scale every score to fall between 0 and 1 so that all tags count equally.  Next, we turn each book and each movie into a long, sparse vector of its tag scores.  To make those easier to compare, we run PCA, which squeezes each vector down to 40 numbers.  We build two fast “hash” indexes—one on movies and one on books—and for each book we look up its 10 nearest movies, and for each movie we look up its 10 nearest books.  We double-check that every item got at least 10 recommendations.  Then we convert the raw distances into a 0–1 similarity score, and finally save everything as two JSON files: one that maps each book to 10 movies, and another that maps each movie to 10 books.

## Set up

In [1]:
# Import the findspark module 
import findspark

# Initialize via the full spark path
findspark.init("/usr/local/spark/")
#findspark.init("C:/spark/")

In [2]:
# Import the SparkSession module
from pyspark.sql import SparkSession

# Import the collections module
import collections

# Gets an existing :class:`SparkSession` or, if there is no existing one, creates a
# new one based on the options set in this builder.
spark = SparkSession.builder \
   .master("local[8]") \
   .appName("Project-cross-domain") \
   .getOrCreate()

# Main entry point for Spark functionality. A SparkContext represents the
# connection to a Spark cluster, and can be used to create :class:`RDD` and
# broadcast variables on that cluster.
sc = spark.sparkContext

In [3]:
# Imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, min as min_, max as max_, lit, collect_list, struct, row_number, size
)
from pyspark.sql.functions import min as _min, max as _max, col, lit, expr
from pyspark.sql.window import Window
from pyspark.ml.feature import PCA, BucketedRandomProjectionLSH
from pyspark.ml.linalg import SparseVector, Vectors, VectorUDT
from pyspark.sql.types import FloatType
from pyspark.sql.functions import udf
from pyspark.ml.clustering import LDA

### Short Summary for CF
1. Loading of the data
2. Normalizing the scores to be between 0 and 1  
3. Transforming the data into a form compatible with the model
4. Model Training (PCA + LSH)
 - a. Principal Component Analysis
 - b. LSH for books -> movie recommendations
 - c. LSH for movies -> book recommendations
5. Checking if all items have at least 10 recommendations
6. Normalizing the final scores to be between 0 and 1
7. Saving the files

#### 1. Loading of the data

In [4]:
# Loading book_tagdl_filtered.csv
try: 
    book_tags = spark.read.csv("./cross_domain_data/book_tagdl_filtered.csv", header=True, inferSchema=True) \
               .withColumn("score", col("score").cast("double"))
except Exception as e:
    print(f"Error loading file: {e}")
    raise

# Loading movie_tagdl_filtered_1980_plus.csv
try: 
    movie_tags = spark.read.csv("./cross_domain_data/movie_tagdl_filtered_1980_plus.csv", header=True, inferSchema=True) \
               .withColumn("score", col("score").cast("double"))
except Exception as e:
    print(f"Error loading file: {e}")
    raise

# Loading matching_filtered.csv
try: 
    matching = spark.read.csv("./cross_domain_data/matching_filtered.csv", header=True, inferSchema=True)
except Exception as e:
    print(f"Error loading file: {e}")
    raise

#### 2. Normalizing the scores to be between 0 and 1

In [5]:
# Compute global min and max of raw scores
all_scores = book_tags.select("score").union(movie_tags.select("score"))
stats = all_scores.agg(
    min_("score").alias("min_score"),
    max_("score").alias("max_score")
).first()
min_score, max_score = stats["min_score"], stats["max_score"]

# Normalize scores to [0,1] with SQL expressions
book_norm = book_tags.withColumn(
    "norm_score",
    (col("score") - lit(min_score)) / lit(max_score - min_score)
).select("item_id", "tag", "norm_score")

movie_norm = movie_tags.withColumn(
    "norm_score",
    (col("score") - lit(min_score)) / lit(max_score - min_score)
).select("item_id", "tag", "norm_score")

#### 3. Transforming the data into a form compatible with the model

In [6]:
# Align into unified tags + item_type
books_aligned = book_norm.join(
    matching, book_norm.tag == matching.book_tag, "inner"
).select(
    col("item_id"),
    col("movie_tag").alias("unified_tag"),
    col("norm_score")
).withColumn("item_type", lit("book"))

movies_aligned = movie_norm.join(
    matching, movie_norm.tag == matching.movie_tag, "inner"
).select(
    col("item_id"),
    col("movie_tag").alias("unified_tag"),
    col("norm_score")
).withColumn("item_type", lit("movie"))

all_items = books_aligned.union(movies_aligned)

# Build vocabulary & UDF for sparse vectors
unique_tags = all_items.select("unified_tag").distinct().rdd.map(lambda r: r[0]).collect()
tag_to_index = {t:i for i,t in enumerate(sorted(unique_tags))}
vocab_size = len(tag_to_index)

@udf(VectorUDT())
def build_vector(pairs):
    idx_vals = [(tag_to_index[t], float(s)) for t,s in pairs if t in tag_to_index]
    idx_vals.sort(key=lambda x: x[0])
    if not idx_vals:
        return SparseVector(vocab_size, [], [])
    idxs, vals = zip(*idx_vals)
    return SparseVector(vocab_size, list(idxs), list(vals))

items_tagscore = all_items.groupBy("item_id","item_type") \
    .agg(collect_list(struct("unified_tag","norm_score")).alias("pairs"))

items_vec = items_tagscore.withColumn(
    "features", build_vector(col("pairs"))
).select("item_id","item_type","features")

## 4. Model Training (PCA + LSH) - Content-based Filtering

#### 4a. Principal Component Analysis

In [7]:
# PCA to 40 dimensions
pca = PCA(k=40, inputCol="features", outputCol="pcaFeatures")
pca_model = pca.fit(items_vec)
items_pca = pca_model.transform(items_vec) \
    .select("item_id","item_type", col("pcaFeatures").alias("features"))


# Split into books & movies
books  = items_pca.filter(col("item_type")=="book")  \
         .select(col("item_id").alias("book_id"), "features")
movies = items_pca.filter(col("item_type")=="movie") \
         .select(col("item_id").alias("movie_id"), "features")

#### 4b. LSH for books -> movie recommendations

In [8]:
# kNN via LSH (k=10) with no cutoff
k = 10

# LSH index on movies, query with books
lsh_movies = BucketedRandomProjectionLSH(
    inputCol="features", outputCol="hashes",
    bucketLength=1.0, numHashTables=10, seed=42
).fit(movies)

sim_bm = lsh_movies.approxSimilarityJoin(books, movies, 2.0, "distance")
w_b = Window.partitionBy("datasetA.book_id").orderBy(col("distance"))
top_movies_per_book = sim_bm.withColumn("rank", row_number().over(w_b)) \
    .filter(col("rank") <= k) \
    .select(
        col("datasetA.book_id").alias("book_id"),
        col("datasetB.movie_id").alias("movie_id"),
        col("distance")
    ).groupBy("book_id") \
    .agg(collect_list(struct("movie_id","distance")).alias("recommended_movies"))


#### 4c. LSH for movies -> book recommendations

In [9]:
# LSH index on books, query with movies
lsh_books = BucketedRandomProjectionLSH(
    inputCol="features", outputCol="hashes",
    bucketLength=1.0, numHashTables=10, seed=42
).fit(books)

sim_mb = lsh_books.approxSimilarityJoin(movies, books, 2.0, "distance")
w_m = Window.partitionBy("datasetA.movie_id").orderBy(col("distance"))
top_books_per_movie = sim_mb.withColumn("rank", row_number().over(w_m)) \
    .filter(col("rank") <= k) \
    .select(
        col("datasetA.movie_id").alias("movie_id"),
        col("datasetB.book_id").alias("book_id"),
        col("distance")
    ).groupBy("movie_id") \
    .agg(collect_list(struct("book_id","distance")).alias("recommended_books"))

In [10]:
# Show results
top_movies_per_book.show(5, truncate=False)
top_books_per_movie.show(5, truncate=False)

+-------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|book_id|recommended_movies                                                                                                                                                                                                                                                                       |
+-------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|31912  |[[5761, 1.2071445599214623], [715, 1.2076243413619303], [1977, 1.2151688891397823], [3892, 1.2214477341044971], [72

#### 5. Checking if all items have at least 10 recommendations

In [11]:
# Total number of books & movies in your catalog
total_books = books.select("book_id").distinct().count()
total_movies = movies.select("movie_id").distinct().count()
print(f"Total books = {total_books}, total movies = {total_movies}")

# For each book, compute how many recs it actually got
books_rec_count = top_movies_per_book.withColumn("num_recs", size(col("recommended_movies")))

# Find books with fewer than 10
books_missing = books_rec_count.filter(col("num_recs") < 10)
missing_books_count = books_missing.count()

if missing_books_count == 0:
    print("✅ Every book has at least 10 movie recommendations")
else:
    print(f"{missing_books_count} books have fewer than 10 recs (out of {total_books}):")

Total books = 9374, total movies = 4061
1816 books have fewer than 10 recs (out of 9374):


In [12]:
# Repeating the same for movies
movies_rec_count = top_books_per_movie.withColumn("num_recs", size(col("recommended_books")))
movies_missing = movies_rec_count.filter(col("num_recs") < 10)
missing_movies_count = movies_missing.count()

if missing_movies_count == 0:
    print("✅ Every movie has at least 10 book recommendations")
else:
    print(f"{missing_movies_count} movies have fewer than 10 recs (out of {total_movies}):")
    movies_missing.show(truncate=False)

25 movies have fewer than 10 recs (out of 4061):
+--------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+
|movie_id|recommended_books                                                                                                                                                                                                                                                                         |num_recs|
+--------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+
|33722   |[[44960981, 1.9229266315154807],

#### 6. Normalizing the final scores to be between 0 and 1

In [13]:
# Compute global min & max distance (book -> movie)
stats_bm = sim_bm.agg(
    _min("distance").alias("min_d"),
    _max("distance").alias("max_d")
).first()
min_d_bm, max_d_bm = stats_bm["min_d"], stats_bm["max_d"]

# Normalize the distances into a [0,1] score
# transform(recommended_movies, x -> struct(x.movie_id, new_score))
top_movies_per_book_norm = top_movies_per_book.withColumn(
    "recommended_movies",
    expr(f"""
      transform(
        recommended_movies,
        x -> struct(
          x.movie_id   AS movie_id,
          (1.0 - (x.distance - {min_d_bm})/({max_d_bm} - {min_d_bm})) AS score
        )
      )
    """)
)

# Do the same for movies -> books
stats_mb = sim_mb.agg(
    _min("distance").alias("min_d"),
    _max("distance").alias("max_d")
).first()
min_d_mb, max_d_mb = stats_mb["min_d"], stats_mb["max_d"]

top_books_per_movie_norm = top_books_per_movie.withColumn(
    "recommended_books",
    expr(f"""
      transform(
        recommended_books,
        x -> struct(
          x.book_id    AS book_id,
          (1.0 - (x.distance - {min_d_mb})/({max_d_mb} - {min_d_mb})) AS score
        )
      )
    """)
)

# Inspect
print("Books -> Movies with normalized [0,1] scores:")
top_movies_per_book_norm.show(3, truncate=False)

print("Movies -> Books with normalized [0,1] scores:")
top_books_per_movie_norm.show(3, truncate=False)

Books -> Movies with normalized [0,1] scores:
+-------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|book_id|recommended_movies                                                                                                                                                                                                                                                                     |
+-------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|31912  |[[5761, 0.5895721476473605], [715, 0.5892153790457564], [1977, 0.5836052042

#### 7. Saving the files

In [ ]:
top_movies_per_book_norm.coalesce(1).write.mode("overwrite").json("output/books_to_movies.json")
top_books_per_movie_norm.coalesce(1).write.mode("overwrite").json("output/movies_to_books.json")